# Cuaderno U2-04. Visualización e interpretación preliminar de resultados

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2
**Unidad 2.** Herramientas computacionales para modelación y simulación
**Subtema del plan.** 2.4 Visualización e interpretación preliminar de resultados
**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad2/U2_04_visualizacion_e_interpretacion.ipynb)

La insignia anterior queda con la dirección del repositorio pendiente. El
docente reemplaza `msc-unisucre/msc2026-material` por la ruta real antes de publicar.

Este cuaderno recorre la Sección 2.5 del libro. La primera figura de
una simulación no se hace para el informe, se hace para uno mismo, y su
propósito es detectar lo que las tablas esconden. Aquí se construyen figuras
de diagnóstico, se reproducen las cuatro decisiones gráficas que distorsionan
la lectura de la Figura 2.12 y se desarrolla el Ejemplo 2.6 con la Tabla 2.5
completa.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante debe ser capaz de lo siguiente.

1. Programar sobre el objeto de ejes y no sobre la interfaz de estado global, de modo que una función de dibujo sea componible.
2. Producir una figura de diagnóstico con magnitud, unidad, límites explícitos y referencia física, como el Listado 2.12.
3. Reconocer las cuatro decisiones gráficas de la Figura 2.12 que distorsionan la lectura y aplicar su corrección.
4. Aplicar en orden las cinco comprobaciones de lectura preliminar y rechazar un resultado que viole una cota física.
5. Reproducir la Tabla 2.5 del libro y explicar por qué un exceso del cinco por ciento sobre el límite de estabilidad destruye la solución.

## Puesta a punto

La primera celda instala lo que falte y la segunda fija la semilla del curso,
la paleta del libro y la función que compara cada resultado con el valor
publicado. Ningún resultado de este cuaderno depende de una ejecución
concreta.

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        *faltantes], check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
# Configuración común a todos los cuadernos del curso.
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
          "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110,
                     "font.size": 9, "axes.grid": True,
                     "grid.linewidth": 0.4, "grid.alpha": 0.5,
                     "axes.prop_cycle": plt.cycler(color=list(PALETA.values()))})


def contra_libro(nombre: str, calculado: float, publicado: float,
                 unidad: str = "", tol: float = 1e-3, relativa: bool = True,
                 exigir: bool = True, nota: str = "") -> None:
    """Compara un resultado del cuaderno con el valor que publica el libro.

    Detiene la ejecución si la diferencia excede la tolerancia y `exigir` es
    verdadero. Las magnitudes que dependen de la máquina, como los tiempos de
    ejecución, se informan con `exigir=False` y una nota que lo advierte.
    """
    error = abs(calculado - publicado)
    if relativa and publicado != 0.0:
        error = error / abs(publicado)
    ok = error <= tol
    print(f"{nombre:<44s} cuaderno {calculado:>13.6g}  "
          f"libro {publicado:>13.6g} {unidad:<9s} "
          f"{'coincide' if ok else 'DIFIERE '}{nota}")
    if exigir and not ok:
        raise AssertionError(
            f"{nombre}, el cuaderno da {calculado!r} y el libro publica "
            f"{publicado!r}, con error {error:.3e}")


print("NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| pandas", pd.__version__, "| SymPy", sp.__version__)

## 1. La gramática de Matplotlib

La Figura 2.11 del libro organiza toda figura en tres niveles, que son el
lienzo, el sistema de ejes y los artistas que lo pueblan. La consecuencia
práctica es que conviene programar sobre el objeto de ejes, pues puede pasarse
como argumento, reutilizarse en una subgráfica y probarse. Una función que
recibe un sistema de ejes es componible, una que dibuja en la figura activa no
lo es.

In [ ]:
def dibujar_perfil(ax, t, T, etiqueta, color, estilo="-"):
    """Dibuja un perfil térmico en el sistema de ejes recibido."""
    ax.plot(t / 60.0, T, estilo, color=color, lw=1.4, label=etiqueta)
    ax.set_xlabel("Tiempo desde el inicio del proceso (min)")
    ax.set_ylabel("Temperatura en el centro de la lata (grados Celsius)")
    return ax


t_dem = np.arange(0.0, 2401.0, 11.0)
T_dem = 121.0 - 96.0 * np.exp(-t_dem / 1450.0)

fig, ejes = plt.subplots(1, 2, figsize=(13.5 / 2.54, 5.4 / 2.54),
                         layout="constrained")
dibujar_perfil(ejes[0], t_dem, T_dem, "proceso completo", PALETA["azul"])
dibujar_perfil(ejes[1], t_dem[:80], T_dem[:80], "primeros 15 min",
               PALETA["verde"])
for ax in ejes:
    ax.legend(loc="lower right", fontsize=7.5)

print("la figura contiene", len(fig.axes), "sistemas de ejes")
print("el primero contiene", len(ejes[0].lines), "artista de tipo línea")
plt.show()

## 2. Una figura de diagnóstico

El Listado 2.12 del libro toma cuatro decisiones deliberadas. Los ejes llevan
su magnitud y su unidad, la referencia física relevante aparece como una línea
horizontal que permite juzgar el resultado sin calcular, los límites se fijan
de manera explícita para que la figura no cambie de escala al cambiar los
datos, y la exportación se hace en formato vectorial.

In [ ]:
# Listado 2.12 del libro, con la exportación dirigida a una carpeta relativa.
SALIDA = Path("salida") / "figuras"
SALIDA.mkdir(parents=True, exist_ok=True)

t = np.arange(0.0, 2401.0, 11.0)
T = 121.0 - 96.0 * np.exp(-t / 1450.0)

fig, ax = plt.subplots(figsize=(13.5 / 2.54, 7.0 / 2.54),
                       layout="constrained")
ax.plot(t / 60.0, T, color=PALETA["azul"], lw=1.4, label="paso estable")
ax.axhline(121.0, color=PALETA["gris"], lw=0.8, ls="--")   # medio calefactor
ax.set_xlabel("Tiempo desde el inicio del proceso (min)")
ax.set_ylabel("Temperatura en el centro de la lata (grados Celsius)")
ax.set_xlim(0.0, 40.0)
ax.set_ylim(20.0, 130.0)
ax.legend(loc="lower right")
fig.savefig(SALIDA / "centro_lata.pdf")
plt.show()

print("archivo exportado:", (SALIDA / "centro_lata.pdf").as_posix(),
      "de", (SALIDA / "centro_lata.pdf").stat().st_size, "bytes")
print("la referencia de 121 grados Celsius permite juzgar el resultado sin "
      "calcular, pues ninguna curva puede cruzarla")

## 3. Cuatro decisiones que distorsionan la lectura

La Figura 2.12 del libro reúne cuatro decisiones gráficas que se cometen casi
siempre sin intención de engañar y que sobreviven al escrutinio porque la
figura se ve profesional. El segundo par merece comentario aparte, pues el
libro publica una correlación bruta de 0.923 entre dos series que solo
comparten estacionalidad, y una correlación de los residuos de apenas 0.212
tras retirar la componente anual, valor que con treinta y seis observaciones
no sostiene ninguna afirmación.

In [ ]:
generador = np.random.default_rng(SEMILLA)

mes = np.arange(1, 37)
estacion = np.sin(2 * np.pi * mes / 12.0)
consumo = 128.0 + 9.0 * estacion + generador.normal(0, 2.6, mes.size)
turbiedad = 5.4 + 1.7 * estacion + generador.normal(0, 0.50, mes.size)


def residuo_estacional(y, mes):
    """Resta la componente anual ajustada por mínimos cuadrados."""
    base = np.column_stack([np.ones(mes.size),
                            np.sin(2 * np.pi * mes / 12.0),
                            np.cos(2 * np.pi * mes / 12.0)])
    coef, *_ = np.linalg.lstsq(base, y, rcond=None)
    return y - base @ coef


r_bruto = float(np.corrcoef(consumo, turbiedad)[0, 1])
res_c = residuo_estacional(consumo, mes)
res_t = residuo_estacional(turbiedad, mes)
r_residuos = float(np.corrcoef(res_c, res_t)[0, 1])

contra_libro("correlación con la estacionalidad", r_bruto, 0.923, "",
             tol=5e-4, relativa=False)
contra_libro("correlación de los residuos", r_residuos, 0.212, "",
             tol=5e-4, relativa=False)

In [ ]:
fig, ejes = plt.subplots(4, 2, figsize=(13.5 / 2.54, 15.5 / 2.54),
                         layout="constrained")

# (a) eje vertical truncado
bombas = ["B1", "B2", "B3", "B4"]
eta = np.array([0.872, 0.881, 0.886, 0.893])
for k, ax in enumerate(ejes[0]):
    ax.bar(bombas, eta, color=PALETA["azul"], width=0.55)
    ax.set_ylabel("Eficiencia (adimensional)")
    ax.set_xlabel("Bomba del sistema de rebombeo")
    ax.set_ylim(0.865, 0.897) if k == 0 else ax.set_ylim(0.0, 1.0)
ejes[0, 0].set_title("(a) eje vertical truncado", fontsize=8.2)
ejes[0, 1].set_title("el mismo dato con el eje desde cero", fontsize=8.2)

# (b) doble eje ajustado
ax = ejes[1, 0]
ax.plot(mes, consumo, color=PALETA["azul"], lw=1.2)
ax.set_ylabel("Consumo (L/hab por día)", color=PALETA["azul"], fontsize=8.0)
gemelo = ax.twinx()
gemelo.plot(mes, turbiedad, color=PALETA["rojo"], lw=1.2)
gemelo.set_ylabel("Turbiedad (NTU)", color=PALETA["rojo"], fontsize=8.0)
gemelo.grid(False)
ax.set_xlabel("Mes de operación")
ax.set_title(f"(b) doble eje ajustado, r = {r_bruto:.3f}", fontsize=8.2)

ax = ejes[1, 1]
ax.plot(res_c, res_t, "o", ms=3.6, color=PALETA["gris"])
ax.axhline(0.0, color=PALETA["gris"], lw=0.6)
ax.axvline(0.0, color=PALETA["gris"], lw=0.6)
ax.set_xlabel("Residuo de consumo (L/hab por día)")
ax.set_ylabel("Residuo de turbiedad (NTU)")
ax.set_title(f"residuos sin estacionalidad, r = {r_residuos:.3f}",
             fontsize=8.2)

# (c) escala transformada sin declarar
tt = np.linspace(0.0, 60.0, 40)
conteo = 1.0e6 * np.exp(-0.115 * tt) * np.exp(generador.normal(0, 0.06,
                                                               tt.size))
ax = ejes[2, 0]
ax.plot(tt, np.log10(conteo), "o-", ms=3.0, color=PALETA["verde"])
ax.set_xlabel("Tiempo de tratamiento (min)")
ax.set_ylabel("Recuento (UFC/mL)")
ax.set_title("(c) escala transformada sin declarar", fontsize=8.2)

ax = ejes[2, 1]
ax.semilogy(tt, conteo, "o-", ms=3.0, color=PALETA["verde"])
ax.set_xlabel("Tiempo de tratamiento (min)")
ax.set_ylabel("Recuento (UFC/mL)")
ax.set_title("eje logarítmico rotulado", fontsize=8.2)

# (d) promedio sin dispersión
dosis = np.array([0.0, 2.0, 4.0, 6.0, 8.0])
medias = np.array([18.4, 17.1, 16.2, 15.9, 15.7])
dispersion = np.array([2.6, 2.9, 3.1, 3.0, 3.2])
nube = [m + s * generador.standard_normal(12)
        for m, s in zip(medias, dispersion)]
ax = ejes[3, 0]
ax.plot(dosis, medias, "o-", color=PALETA["morado"])
ax.set_xlabel("Dosis de coagulante (mg/L)")
ax.set_ylabel("DBO final (mg/L)")
ax.set_ylim(15.0, 19.0)
ax.set_title("(d) solo el promedio", fontsize=8.2)

ax = ejes[3, 1]
for d, muestra in zip(dosis, nube):
    ax.plot(np.full(muestra.size, d), muestra, "o", ms=2.8,
            color=PALETA["gris"], alpha=0.55)
ax.errorbar(dosis, medias, yerr=dispersion, fmt="o-", ms=4.0, lw=1.3,
            capsize=2.5, color=PALETA["morado"])
ax.set_xlabel("Dosis de coagulante (mg/L)")
ax.set_ylabel("DBO final (mg/L)")
ax.set_title("promedio, dispersión y datos", fontsize=8.2)

for ax in ejes.ravel():
    ax.tick_params(labelsize=7.5)
    ax.xaxis.label.set_size(8.0)
    ax.yaxis.label.set_size(8.0)
plt.show()

In [ ]:
diferencia = float(eta.max() - eta.min())
print(f"diferencia real entre la mejor y la peor bomba {diferencia:.3f}, "
      f"esto es {100 * diferencia:.1f} puntos de eficiencia")
print(f"con el eje entre 0.865 y 0.897 esa diferencia ocupa el "
      f"{100 * diferencia / (0.897 - 0.865):.0f} por ciento de la altura, "
      "y con el eje desde cero ocupa el "
      f"{100 * diferencia / 1.0:.1f} por ciento")
print("\nla diferencia entre las series sin estacionalidad es "
      f"{r_bruto / r_residuos:.1f} veces menor que la aparente, y con "
      f"{mes.size} observaciones un coeficiente de {r_residuos:.3f} no "
      "sostiene ninguna afirmación de causalidad")

## 4. Las cinco comprobaciones de lectura preliminar

El libro fija un orden para leer una figura de simulación. Se comprueba
primero el orden de magnitud frente al valor esperado del proceso, después el
signo y el sentido de la tendencia, y luego los extremos, es decir, el valor
inicial y el asintótico. La cuarta comprobación rechaza el resultado si alguna
variable excede una cota física del sistema, y la quinta lo rechaza si
aparecen oscilaciones de periodo igual al doble del paso de integración. Solo
si las cinco se superan tiene sentido repetir el cálculo con la malla
refinada.

In [ ]:
def revisar_resultado(t, campo, esperado, cota_inferior, cota_superior,
                      valor_inicial, valor_asintotico, tol_extremos=1.0):
    """Aplica en orden las cinco comprobaciones de la Sección 2.5."""
    centro = campo[:, 0]
    informe = {}
    informe["1 orden de magnitud"] = bool(
        0.1 * esperado <= np.abs(centro).max() <= 10.0 * esperado)
    informe["2 sentido de la tendencia"] = bool(
        np.median(np.diff(centro)) > 0.0)
    informe["3 extremos"] = bool(
        abs(centro[0] - valor_inicial) <= tol_extremos
        and abs(centro[-1] - valor_asintotico) <= tol_extremos)
    informe["4 cota física"] = bool(
        campo.min() >= cota_inferior and campo.max() <= cota_superior)
    diferencias = np.diff(campo, axis=0)
    cambios = np.sign(diferencias[1:]) != np.sign(diferencias[:-1])
    informe["5 sin oscilación de periodo doble"] = bool(
        cambios.mean() < 0.25)
    return pd.Series(informe)

## 5. Ejemplo 2.6, lectura preliminar de un perfil
térmico

El centro de una conserva de 40 mm de espesor, con difusividad térmica de
1.4e-7 m2/s, se simula mediante un esquema explícito de diferencias finitas
sobre once nodos, a partir de 25 grados Celsius y con las superficies
mantenidas a 121 grados Celsius. Un analista escoge un paso de tiempo de 15 s
por comodidad de registro. El libro publica un número de Fourier de malla de
0.525 con ese paso y de 0.385 con el paso de referencia de 11 s, frente al
límite de 0.5 que impone la estabilidad, alcanzado en 14.286 s.

In [ ]:
ALFA = 1.4e-7          # m2/s
SEMIESPESOR = 0.020    # m, media placa por simetría
N_NODOS = 11
T_INICIAL, T_MEDIO = 25.0, 121.0     # grados Celsius
DX = SEMIESPESOR / (N_NODOS - 1)


def simular(dt: float, t_final: float = 1800.0):
    """Esquema explícito con simetría en el plano medio, Ecuación 2.6."""
    Fo = ALFA * dt / DX ** 2
    T = np.full(N_NODOS, T_INICIAL)
    T[-1] = T_MEDIO
    tiempos, campo = [0.0], [T.copy()]
    t_actual = 0.0
    while t_actual < t_final - 1e-9:
        nuevo = T.copy()
        nuevo[0] = T[0] + 2.0 * Fo * (T[1] - T[0])      # simetría
        nuevo[1:-1] = T[1:-1] + Fo * (T[2:] - 2.0 * T[1:-1] + T[:-2])
        nuevo[-1] = T_MEDIO
        T = nuevo
        t_actual += dt
        tiempos.append(t_actual)
        campo.append(T.copy())
    return Fo, np.array(tiempos), np.array(campo)


Fo_11, t_11, campo_11 = simular(11.0)
Fo_15, t_15, campo_15 = simular(15.0)
dt_limite = 0.5 * DX ** 2 / ALFA

print(f"espaciamiento de la malla {DX * 1000:.0f} mm")
contra_libro("número de Fourier con paso de 11 s", Fo_11, 0.385, "",
             tol=5e-4, relativa=False)
contra_libro("número de Fourier con paso de 15 s", Fo_15, 0.525, "",
             tol=5e-4, relativa=False)
contra_libro("paso límite de estabilidad", dt_limite, 14.286, "s",
             tol=5e-4, relativa=False)
print(f"el paso elegido excede el límite en un "
      f"{100 * (15.0 - dt_limite) / dt_limite:.1f} por ciento")

La Tabla 2.5 del libro recoge la temperatura calculada
en el centro y en el nodo situado a 6 mm del plano medio, con el paso estable
y con el paso que excede el límite. La celda siguiente la reproduce
completa.

In [ ]:
def en_minuto(t, campo, minuto, nodo=0):
    """Valor del paso más cercano al minuto pedido, como en la Tabla 2.5."""
    k = int(np.argmin(np.abs(t - minuto * 60.0)))
    return float(campo[k, nodo])


def en_minuto_interp(t, campo, minuto, nodo=0):
    """Valor interpolado al minuto exacto, para el estudio de refinamiento."""
    return float(np.interp(minuto * 60.0, t, campo[:, nodo]))


MINUTOS = (5, 10, 15, 20, 25)
PUBLICADO = {
    5: (30.41, 32.80, 34.50), 10: (49.08, 68.85, 37.68),
    15: (65.17, 177.37, -29.48), 20: (77.81, 678.30, -452.85),
    25: (87.60, 3279.65, -2752.95)}

filas = []
for minuto in MINUTOS:
    filas.append({"minuto": minuto,
                  "centro dt=11 s": en_minuto(t_11, campo_11, minuto, 0),
                  "centro dt=15 s": en_minuto(t_15, campo_15, minuto, 0),
                  "6 mm dt=15 s": en_minuto(t_15, campo_15, minuto, 3)})
tabla_24 = pd.DataFrame(filas)
print(tabla_24.round(2).to_string(index=False))
print()
for minuto in MINUTOS:
    fila = tabla_24.loc[tabla_24["minuto"] == minuto].iloc[0]
    esperado = PUBLICADO[minuto]
    contra_libro(f"centro a {minuto:2d} min con paso de 11 s",
                 fila["centro dt=11 s"], esperado[0], "gC", tol=0.005,
                 relativa=False)
    contra_libro(f"centro a {minuto:2d} min con paso de 15 s",
                 fila["centro dt=15 s"], esperado[1], "gC", tol=0.005,
                 relativa=False)
    contra_libro(f"nodo de 6 mm a {minuto:2d} min con paso de 15 s",
                 fila["6 mm dt=15 s"], esperado[2], "gC", tol=0.005,
                 relativa=False)

La cuarta comprobación rechaza el resultado en cuanto
algún nodo excede la temperatura del medio. El libro publica que eso ocurre por
primera vez a los 795 s, esto es, a los 13.25 min de proceso. La quinta lo
habría rechazado antes, pues los nodos oscilan con periodo igual al doble del
paso desde las primeras iteraciones.

In [ ]:
interiores = campo_15[:, :-1]
primer_exceso = int(np.argmax(interiores.max(axis=1) > T_MEDIO))
t_exceso = float(t_15[primer_exceso])

contra_libro("primer instante con exceso sobre el medio", t_exceso, 795.0,
             "s", tol=0.5, relativa=False)
contra_libro("primer exceso en minutos", t_exceso / 60.0, 13.25, "min",
             tol=0.01, relativa=False)

informe_11 = revisar_resultado(t_11, campo_11, esperado=100.0,
                               cota_inferior=T_INICIAL - 1.0,
                               cota_superior=T_MEDIO + 1e-6,
                               valor_inicial=25.0, valor_asintotico=121.0,
                               tol_extremos=35.0)
informe_15 = revisar_resultado(t_15, campo_15, esperado=100.0,
                               cota_inferior=T_INICIAL - 1.0,
                               cota_superior=T_MEDIO + 1e-6,
                               valor_inicial=25.0, valor_asintotico=121.0,
                               tol_extremos=35.0)
print()
print(pd.DataFrame({"paso 11 s": informe_11,
                    "paso 15 s": informe_15}).to_string())
assert informe_11.all(), "el paso estable debe superar las cinco comprobaciones"
assert not informe_15["4 cota física"], "el paso inestable viola la cota física"
assert not informe_15["5 sin oscilación de periodo doble"], \
    "el paso inestable oscila con periodo doble"

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13.5 / 2.54, 6.0 / 2.54),
                         layout="constrained")
ax = ejes[0]
ax.plot(t_11 / 60.0, campo_11[:, 0], color=PALETA["azul"], lw=1.4,
        label="paso estable de 11 s")
ax.plot(t_15 / 60.0, campo_15[:, 0], color=PALETA["rojo"], lw=1.2, ls="--",
        label="paso de 15 s")
ax.axhline(T_MEDIO, color=PALETA["gris"], lw=0.8, ls=":")
ax.axvline(t_exceso / 60.0, color=PALETA["naranja"], lw=0.8)
ax.set_xlabel("Tiempo (min)")
ax.set_ylabel("Temperatura del centro (grados Celsius)")
ax.set_xlim(0.0, 25.0)
ax.set_ylim(0.0, 200.0)
ax.legend(loc="upper left", fontsize=7.5)

ax = ejes[1]
ventana = (t_15 <= 300.0)
ax.plot(t_15[ventana], campo_15[ventana, 3], "o-", ms=3.0,
        color=PALETA["rojo"], label="nodo de 6 mm, paso de 15 s")
ax.plot(t_11[t_11 <= 300.0], campo_11[t_11 <= 300.0, 3], "-",
        color=PALETA["azul"], lw=1.4, label="nodo de 6 mm, paso de 11 s")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Temperatura (grados Celsius)")
ax.legend(loc="lower right", fontsize=7.5)
plt.show()

print("el panel derecho muestra la oscilación de periodo igual al doble del "
      "paso, que es la quinta comprobación y la que atrapa el artefacto antes "
      "de que la temperatura llegue a valores absurdos")

## 6. Refinamiento del paso

Superadas las cinco comprobaciones, el libro exige repetir el cálculo con la
malla o el paso refinados a la mitad y aceptar el resultado únicamente cuando
el cambio queda por debajo de la tolerancia declarada. La celda siguiente lo
hace con el paso estable.

In [ ]:
pasos = [11.0, 5.5, 2.75, 1.375]
referencia = None
filas = []
for dt in pasos:
    _, t_r, campo_r = simular(dt, t_final=1500.0)
    valor = en_minuto_interp(t_r, campo_r, 25, 0)
    cambio = np.nan if referencia is None else abs(valor - referencia)
    filas.append({"paso_s": dt, "centro_25min": valor, "cambio": cambio})
    referencia = valor
print(pd.DataFrame(filas).round(5).to_string(index=False))

TOLERANCIA = 0.03      # grados Celsius
ultimo = filas[-1]["cambio"]
print(f"\ncambio al último refinamiento {ultimo:.5f} grados Celsius, "
      f"tolerancia declarada {TOLERANCIA:.2f}")
assert ultimo < TOLERANCIA, "el resultado todavía depende del paso"
print("el cambio se reduce a la mitad cada vez que el paso se reduce a la "
      "mitad, que es el orden uno en el tiempo del esquema explícito. "
      "El resultado ya es independiente del paso dentro de la tolerancia "
      "declarada, de modo que puede aceptarse")

## 7. Ejercicios guiados

Cinco celdas incompletas con su verificación inmediatamente después.

### Ejercicio 1. Una función de dibujo componible

Escriba la función que recibe un sistema de ejes y dibuja en él el perfil de
un nodo, con su magnitud, su unidad y la referencia física. La función no debe
crear la figura ni llamar a la interfaz de estado global.

In [ ]:
# COMPLETE: dibuje en el eje recibido la temperatura del nodo indicado, ponga
# los rótulos con unidad, marque la temperatura del medio con una línea
# horizontal y devuelva el eje.
REVISAR_1 = False


def dibujar_nodo(ax, t, campo, nodo, etiqueta, color):
    """Dibuja el perfil de un nodo en el eje recibido."""
    return ax          # <- reemplace por el dibujo completo

In [ ]:
if REVISAR_1:
    fig_ej, ejes_ej = plt.subplots(1, 2, figsize=(13.5 / 2.54, 5.0 / 2.54),
                                   layout="constrained")
    dibujar_nodo(ejes_ej[0], t_11, campo_11, 0, "centro", PALETA["azul"])
    dibujar_nodo(ejes_ej[1], t_11, campo_11, 3, "6 mm", PALETA["verde"])
    for ax in ejes_ej:
        ax.legend(loc="lower right", fontsize=7.5)
    plt.show()
    for ax in ejes_ej:
        assert ax.get_xlabel(), "el eje horizontal debe llevar rótulo"
        assert "grados Celsius" in ax.get_ylabel(), \
            "el eje vertical debe llevar la unidad"
        assert len(ax.lines) >= 2, "falta la línea de referencia del medio"
    print("ejercicio 1 correcto, la función es componible y rotula con unidad")
else:
    print("ejercicio 1 pendiente, complete la celda y ponga REVISAR_1 = True")

### Ejercicio 2. Número de Fourier y límite de
estabilidad

Escriba la función que devuelve el número de Fourier de malla de la Ecuación
2.6 y el paso máximo que respeta la condición de estabilidad.

In [ ]:
# COMPLETE: devuelva la pareja (Fo, dt_maximo) para una difusividad, un paso
# de tiempo y un espaciamiento dados.
REVISAR_2 = False


def fourier_malla(alfa: float, dt: float, dx: float) -> tuple[float, float]:
    """Número de Fourier de malla y paso máximo estable."""
    return 0.0, 0.0     # <- reemplace por la Ecuación 2.6 y su límite

In [ ]:
if REVISAR_2:
    Fo_ej, dt_max_ej = fourier_malla(ALFA, 15.0, DX)
    contra_libro("Fo del ejercicio", Fo_ej, 0.525, "", tol=5e-4,
                 relativa=False)
    contra_libro("paso máximo del ejercicio", dt_max_ej, 14.286, "s",
                 tol=5e-4, relativa=False)
    assert fourier_malla(ALFA, dt_max_ej, DX)[0] <= 0.5 + 1e-12, \
        "en el paso máximo el número de Fourier debe valer exactamente un medio"
    Fo_fino, _ = fourier_malla(ALFA, 15.0, DX / 2.0)
    print(f"al refinar la malla a la mitad el Fo sube a {Fo_fino:.3f}, "
          "de modo que refinar el espacio obliga a refinar el tiempo por "
          "cuatro")
    print("ejercicio 2 correcto")
else:
    print("ejercicio 2 pendiente, complete la celda y ponga REVISAR_2 = True")

### Ejercicio 3. La cuarta comprobación

Escriba la función que devuelve el primer instante en que algún nodo interior
excede una cota física, o un valor nulo si nunca ocurre.

In [ ]:
# COMPLETE: devuelva el primer tiempo en el que algún nodo interior supera la
# cota, y None si el resultado nunca la viola.
REVISAR_3 = False


def primer_exceso_fisico(t, campo, cota):
    """Primer instante en que un nodo interior supera la cota física."""
    return None        # <- reemplace por la búsqueda

In [ ]:
if REVISAR_3:
    inestable = primer_exceso_fisico(t_15, campo_15, T_MEDIO)
    estable = primer_exceso_fisico(t_11, campo_11, T_MEDIO)
    contra_libro("primer exceso del ejercicio", inestable, 795.0, "s",
                 tol=0.5, relativa=False)
    assert estable is None, "el paso estable nunca debe exceder la cota"
    print(f"con paso de 15 s el resultado se rechaza a los "
          f"{inestable / 60:.2f} min, mucho antes de que la temperatura "
          "resulte absurda")
    print("ejercicio 3 correcto")
else:
    print("ejercicio 3 pendiente, complete la celda y ponga REVISAR_3 = True")

### Ejercicio 4. Problema 2-28, el eje truncado

Un informe muestra la eficiencia de cuatro alternativas fotovoltaicas en
barras con el eje entre 0.86 y 0.90. Construya el par de figuras, la engañosa
y la corregida, y calcule la razón entre la diferencia aparente y la real.
Este es el Problema 2-28 del libro.

In [ ]:
# COMPLETE: calcule la razón entre la altura relativa que ocupa la diferencia
# con el eje truncado y la que ocupa con el eje desde cero.
REVISAR_4 = False
eficiencias = np.array([0.868, 0.877, 0.883, 0.891])
limite_inf, limite_sup = 0.86, 0.90
razon_aparente = 1.0      # <- reemplace por el cálculo

In [ ]:
if REVISAR_4:
    print(f"diferencia real {eficiencias.max() - eficiencias.min():.3f}, "
          f"esto es {100 * (eficiencias.max() - eficiencias.min()):.1f} "
          "puntos porcentuales")
    print(f"factor de magnificación del eje truncado {razon_aparente:.0f}")
    assert abs(razon_aparente - 25.0) < 1e-9, \
        "el factor es el inverso del ancho del eje truncado"
    fig_ej, ejes_ej = plt.subplots(1, 2, figsize=(13.5 / 2.54, 5.0 / 2.54),
                                   layout="constrained")
    for ax, (lo, hi), titulo in zip(
            ejes_ej, [(limite_inf, limite_sup), (0.0, 1.0)],
            ["eje truncado, lectura engañosa", "eje desde cero, lectura fiel"]):
        ax.bar(["A", "B", "C", "D"], eficiencias, color=PALETA["azul"],
               width=0.55)
        ax.set_ylim(lo, hi)
        ax.set_ylabel("Eficiencia del arreglo (adimensional)")
        ax.set_xlabel("Alternativa fotovoltaica")
        ax.set_title(titulo, fontsize=8.2)
    plt.show()
    print("\nun lector desprevenido del panel izquierdo concluiría que la "
          "alternativa D casi duplica a la A, cuando la diferencia real es de "
          "2.3 puntos porcentuales. Con esa diferencia la inversión adicional "
          "solo se justifica si el costo por vatio instalado sube menos de ese "
          "mismo porcentaje")
    print("ejercicio 4 correcto")
else:
    print("ejercicio 4 pendiente, complete la celda y ponga REVISAR_4 = True")

### Ejercicio 5. Estudio de refinamiento

Escriba la función que refina el paso a la mitad hasta que el cambio en la
magnitud de interés quede por debajo de la tolerancia, y devuelva la tabla del
estudio.

In [ ]:
# COMPLETE: refine el paso a la mitad partiendo de dt0 y detenga el estudio
# cuando el cambio quede bajo la tolerancia o se agoten los refinamientos.
REVISAR_5 = False


def estudio_refinamiento(dt0=11.0, tol=0.03, maximo=4):
    """Tabla del estudio de independencia del paso."""
    return pd.DataFrame(columns=["paso_s", "valor", "cambio"])

In [ ]:
if REVISAR_5:
    tabla_ref = estudio_refinamiento()
    print(tabla_ref.round(5).to_string(index=False))
    assert len(tabla_ref) >= 2, "el estudio necesita al menos dos pasos"
    assert float(tabla_ref["cambio"].iloc[-1]) < 0.03, \
        "el estudio debe detenerse cuando el cambio baja de la tolerancia"
    assert tabla_ref["paso_s"].is_monotonic_decreasing, \
        "el paso debe refinarse a la mitad en cada etapa"
    print(f"\nel estudio se detuvo con paso de {tabla_ref['paso_s'].iloc[-1]:.3f} s "
          f"y un cambio de {tabla_ref['cambio'].iloc[-1]:.5f} grados Celsius")
    print("ejercicio 5 correcto")
else:
    print("ejercicio 5 pendiente, complete la celda y ponga REVISAR_5 = True")

## 8. Problemas del capítulo

El Problema 2-28 quedó resuelto como Ejercicio 4. Se aborda aquí el Problema
2-25.

### Problema 2-25

Reproduzca los cuatro pares de la Figura 2.12 con datos de su área y redacte la
frase que un lector desprevenido concluiría de cada versión engañosa.

Los cuatro pares están reproducidos en la Sección 3 de este cuaderno. La celda
siguiente redacta la frase engañosa y la corrección para cada uno, que es la
parte del problema que no se resuelve dibujando.

In [ ]:
lecturas = pd.DataFrame(
    [("(a) eje truncado",
      "la bomba B4 es mucho más eficiente que la B1",
      "las cuatro bombas difieren en dos puntos de eficiencia"),
     ("(b) doble eje ajustado",
      "la turbiedad explica el consumo, con correlación de 0.92",
      "ambas siguen el ciclo anual y su relación directa es de 0.21"),
     ("(c) escala transformada sin declarar",
      "el tratamiento reduce el recuento de manera lineal",
      "el decaimiento es exponencial y el eje está en logaritmo"),
     ("(d) promedio sin dispersión",
      "aumentar la dosis de coagulante reduce la DBO final",
      "la diferencia entre dosis es menor que la dispersión del ensayo")],
    columns=["Panel", "Lo que concluiría un lector desprevenido",
             "Lo que dice el dato"])
for _, fila in lecturas.iterrows():
    print(f"{fila['Panel']}")
    print(f"   engaño     {fila['Lo que concluiría un lector desprevenido']}")
    print(f"   corrección {fila['Lo que dice el dato']}")
print("\nel pie de una figura describe lo que se ve y no lo que se concluye. "
      "Una figura que necesita ser explicada por su autor todavía no está "
      "terminada")

## Cierre

### Lista de comprobación

Al cerrar el cuaderno el estudiante debe poder hacer lo siguiente sin
consultar la solución.

- Escribir una función de dibujo que reciba el sistema de ejes y pueda reutilizarse en cualquier subgráfica.
- Rotular una figura de ingeniería con magnitud, unidad, límites explícitos y la referencia física del proceso.
- Identificar las cuatro distorsiones de la Figura 2.12 en una figura ajena y proponer la corrección de cada una.
- Aplicar en orden las cinco comprobaciones de lectura y decir cuál rechaza un resultado antes que las demás.
- Reproducir la Tabla 2.5 y justificar el límite de estabilidad de un esquema explícito.

### Qué revisar en el libro si algo no salió

- Si la gramática de la figura no salió, la Figura 2.11 y el Listado 2.12.
- Si las figuras engañosas no salieron, la Figura 2.12 y el párrafo que la comenta.
- Si las comprobaciones no salieron, el orden de lectura que fija la Sección 2.5.
- Si el artefacto numérico no salió, la Tabla 2.5, la Ecuación 2.6 y el Ejemplo 2.6.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente automático de programación.
Todo fragmento se sometió al protocolo del Algoritmo 2.3 del libro y cada
resultado numérico se comprueba contra la cifra publicada mediante la función
`contra_libro`. La regla de la asignatura es que el ingeniero responde por el
resultado que firma, con independencia de quién haya tecleado las líneas.